# Lecture 09 — Lab: Random Search and AutoML

**Data Science Specialized Consulting 1**

In Lecture 08, the grid search tested all combinations that you wrote. Today, the budget is **yours**: you decide how many measurements you want (`n_iter`) and draw where to measure. The lab trains the three decisions this creates — **which distribution**, **which range**, and **how many trials** — and ends with AutoML, which does the same for the entire model.

The dataset is the same as in Lecture 08 (`5_musicas.csv`), with the same target (`hit`), the same Pipeline, and the same saved test set. What changes is the search method.

Write your answers in the indicated cells. Any statement mentioning a number must match the number produced in your notebook.

---
## Part 0 — Environment, Data, and Setup

Nothing here is new: it is the conclusion of Lecture 08. The `popularity` column is excluded (it leaks the target — we will return to this in Exercise 11) and the 30% test set remains saved until Exercise 8.

In [71]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import uniform, loguniform, randint
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                     GridSearchCV, RandomizedSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

# Load dataset
mus = pd.read_csv("5_musicas.csv")
y = mus["hit"]
X = pd.get_dummies(mus.drop(columns=["id", "titulo", "artista", "popularidade", "hit"]),
                   columns=["genero"])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def pipe():
    return Pipeline([("esc", StandardScaler()),
                     ("clf", RandomForestClassifier(n_estimators=100, random_state=0))])

print("X:", X.shape, "| hits:", int(y.sum()), f"({y.mean():.1%})")
print("train:", len(X_tr), "| test:", len(X_te), "| hits in test:", int(y_te.sum()))
print("lecture metric: average_precision (AP)")

X: (1050, 18) | hits: 128 (12.2%)
train: 735 | test: 315 | hits in test: 38
lecture metric: average_precision (AP)


---
## Part 1 — Why sample before we sample

Two calculations drive the entire lecture: how much each path costs, and what the probability is that the randomized search finds the target region.

#### Example

**Example.** The cost of the grid search is the product of the number of values of each hyperparameter, multiplied by the validation folds. With 2 values for one parameter, 3 for another, and 5 folds, we have 2 × 3 × 5 = 30 fits.

In [72]:
values = [2, 3]
combos = np.prod(values)
print("combinations:", combos, "| with cv = 5:", combos * 5, "fits")

combinations: 6 | with cv = 5: 30 fits


### Exercise 1 — The Cost of Each Path

The Random Forest model has 5 hyperparameters. A modest grid search uses 4, 5, 4, 4, and 4 values. Calculate:

1. How many combinations the grid search generates and how many fits it costs with `cv = 5`;
2. How many fits a `RandomizedSearchCV` costs with `n_iter = 40` and `cv = 5`;
3. How many fits the grid search would cost if you added **a sixth** hyperparameter with 4 values — and how many the randomized search would cost in the same case.

In [73]:
# Exercise 1 - Calculating search costs

# 1. Grid search combos: 4 * 5 * 4 * 4 * 4 = 1280 combos.
# Total fits with cv=5: 1280 * 5 = 6400 fits.
combos_grid = 4 * 5 * 4 * 4 * 4
fits_grid = combos_grid * 5
print(f"Grid search combinations: {combos_grid} | Total fits (cv=5): {fits_grid}")

# 2. Randomized search fits with n_iter=40 and cv=5: 40 * 5 = 200 fits.
fits_random = 40 * 5
print(f"Randomized search total fits (cv=5, n_iter=40): {fits_random}")

# 3. Cost with a 6th hyperparameter of 4 values:
combos_grid_6 = combos_grid * 4
fits_grid_6 = combos_grid_6 * 5
fits_random_6 = 40 * 5  # Cost remains constant
print(f"6-Parameter Grid search combinations: {combos_grid_6} | Fits: {fits_grid_6}")
print(f"6-Parameter Randomized search total fits: {fits_random_6}")

Grid search combinations: 1280 | Total fits (cv=5): 6400
Randomized search total fits (cv=5, n_iter=40): 200
6-Parameter Grid search combinations: 5120 | Fits: 25600
6-Parameter Randomized search total fits: 200


### **Answer - Exercise 1**

1. **Grid Search Cost:**
   The number of combinations generated by the grid search is the product of the number of values chosen for each hyperparameter:
   $$ 4 \times 5 \times 4 \times 4 \times 4 = 1280 \text{ combinations} $$
   With a cross-validation scheme of $cv = 5$, the total cost is:
   $$ 1280 \times 5 = 6400 \text{ fits} $$

2. **Randomized Search Cost:**
   For `RandomizedSearchCV` with $n\_iter = 40$, the cost is independent of the parameter space volume, depending only on the predefined iterations:
   $$ 40 \times 5 = 200 \text{ fits} $$

3. **Adding a Sixth Hyperparameter:**
   * **Grid Search:** The number of combinations would scale by a factor of 4 to $1280 \times 4 = 5120$, resulting in:
     $$ 5120 \times 5 = 25600 \text{ fits} $$
   * **Randomized Search:** The cost remains completely unchanged, as the budget is capped by $n\_iter$. It would still cost exactly:
     $$ 40 \times 5 = 200 \text{ fits} $$

#### Exemplo

**Exemplo.** Se a região boa ocupa uma fração `p` do espaço, cada sorteio erra com
probabilidade `1 − p`, e `n` sorteios erram todos com `(1 − p)ⁿ`. A chance de acertar ao menos
uma vez é o complemento. Com p = 10 % e n = 10 sorteios:

In [74]:
p, n = 0.10, 10
print(f"chance de errar todos: (1 - {p})^{n} = {(1-p)**n:.4f}")
print(f"chance de acertar ao menos um: {1 - (1-p)**n:.4f}")

chance de errar todos: (1 - 0.1)^10 = 0.3487
chance de acertar ao menos um: 0.6513


### Exercise 2 — The Chance of Randomized Search Finding the Target Region

Build a table of the cumulative probability `1 − (1 − p)ⁿ` for `p` in `[0.05, 0.10, 0.15, 0.25]` and `n` in `[10, 16, 30, 60]`, in percentages.

Then answer: with `p = 15 %`, how many trials (sorteios) do you need to exceed 90%? And what happens to this calculation if the search space has 2, 5, or 50 hyperparameters?

In [75]:
# Hit probability table
ps = [0.05, 0.10, 0.15, 0.25]
sorteios = [10, 16, 30, 60]

print("Hit Probability Table (%):")
print(f"{'n \\\\ p':<6} | " + " | ".join([f"{p*100:>5.0f}%" for p in ps]))
print("-" * 40)
for n_val in sorteios:
    row = []
    for p_val in ps:
        chance = (1 - (1 - p_val)**n_val) * 100
        row.append(f"{chance:>5.1f}%")
    print(f"{n_val:<6} | " + " | ".join(row))

# Find the smallest n for p = 15% to exceed 90%
p_target = 0.15
n_target = 1
while (1 - (1 - p_target)**n_target) < 0.90:
    n_target += 1
print(f"\nSmallest n for p = 15% to exceed 90%: {n_target} (chance: {(1 - (1 - p_target)**n_target)*100:.2f}%)")

Hit Probability Table (%):
n \\ p |     5% |    10% |    15% |    25%
----------------------------------------
10     |  40.1% |  65.1% |  80.3% |  94.4%
16     |  56.0% |  81.5% |  92.6% |  99.0%
30     |  78.5% |  95.8% |  99.2% | 100.0%
60     |  95.4% |  99.8% | 100.0% | 100.0%

Smallest n for p = 15% to exceed 90%: 15 (chance: 91.26%)


In [76]:
# TODO — Table of 1 - (1 - p)**n for each p and each n
# TODO — Find the smallest n that exceeds 90% with p = 0.15

Answer:

In [77]:
# Exercise 2 - Target region hit probabilities
ps = [0.05, 0.10, 0.15, 0.25]
sorteios = [10, 16, 30, 60]

print("Hit Probability Table (%):")
print(f"{'n \\\ p':<6} | " + " | ".join([f"{p*100:>5.0f}%" for p in ps]))
print("-" * 40)
for n_val in sorteios:
    row = []
    for p_val in ps:
        chance = (1 - (1 - p_val)**n_val) * 100
        row.append(f"{chance:>5.1f}%")
    print(f"{n_val:<6} | " + " | ".join(row))

# Calculate exact n trials for p = 15% to exceed 90% probability
p_target = 0.15
n_target = 1
while (1 - (1 - p_target)**n_target) < 0.90:
    n_target += 1
print(f"\nMin trials (n) for p = 15% to pass 90%: {n_target} (cumulative chance: {(1 - (1 - p_target)**n_target)*100:.2f}%)")

Hit Probability Table (%):
n \\ p |     5% |    10% |    15% |    25%
----------------------------------------
10     |  40.1% |  65.1% |  80.3% |  94.4%
16     |  56.0% |  81.5% |  92.6% |  99.0%
30     |  78.5% |  95.8% |  99.2% | 100.0%
60     |  95.4% |  99.8% | 100.0% | 100.0%

Min trials (n) for p = 15% to pass 90%: 15 (cumulative chance: 91.26%)


### **Answer - Exercise 2**

### Hit Probability Table (1 − (1 − p)ⁿ)

| n \ p | 5% | 10% | 15% | 25% |
|---|---|---|---|---|
| **10** | 40.1% | 65.1% | 80.3% | 94.4% |
| **16** | 56.0% | 81.5% | 92.6% | 99.0% |
| **30** | 78.5% | 95.8% | 99.2% | 100.0% |
| **60** | 95.4% | 99.8% | 100.0% | 100.0% |

### Exercise Questions:

1. **With `p = 15%`, how many trials do you need to pass 90%?**
   * A minimum of **15 trials** is required to exceed the 90% threshold, reaching a success probability of **91.26%** (with 14 trials, the chance is only 89.72%).

2. **What happens to this calculation if the search space has 2, 5, or 50 hyperparameters?**
   * **Absolutely nothing changes**. This is a major mathematical advantage of randomized search. The probability of finding the target region depends solely on the relative volume occupied by that region ($p$), and not on the dimensionality of the space. While a grid search suffers from the "curse of dimensionality" (where the number of fits grows exponentially), randomized search maintains the same theoretical coverage efficiency regardless of whether we have 2, 5, or 50 hyperparameters.

---
## Part 2 — RandomizedSearchCV in Practice

Three decisions: which **distribution**, which **range**, and how many **trials**.

#### Example

**Example.** The randomized search dictionary does not take lists of values, but rather **distributions** — objects with a `.rvs()` method. Drawing 5 values from `uniform(0, 1)`:

In [78]:
print(np.round(uniform(0, 1).rvs(5, random_state=0), 3))

[0.549 0.715 0.603 0.545 0.424]


### Exercise 3 — Dictionary Distributions

Draw 2,000 values from each distribution and print the **minimum and maximum** of each sample:

- `uniform(0.1, 0.8)`
- `randint(100, 601)`
- `loguniform(0.01, 100)`

For `loguniform`, also print **what fraction of the draws fell into each order of magnitude** (0.01–0.1 · 0.1–1 · 1–10 · 10–100) and compare it to what `uniform(0.01, 100)` would produce.

Answer: What is the actual range of `uniform(0.1, 0.8)`? And why is hyperparameter `C` in logistic regression (0.01 to 100) sampled using a `loguniform` distribution?

In [79]:
# Exercise 3 - Sampling Distributions and Scales
samples_u = uniform(0.1, 0.8).rvs(2000, random_state=0)
samples_ri = randint(100, 601).rvs(2000, random_state=0)
samples_lu = loguniform(0.01, 100).rvs(2000, random_state=0)

print(f"uniform(0.1, 0.8)     -> Min: {samples_u.min():.4f}, Max: {samples_u.max():.4f}")
print(f"randint(100, 601)     -> Min: {samples_ri.min()}, Max: {samples_ri.max()}")
print(f"loguniform(0.01, 100) -> Min: {samples_lu.min():.4f}, Max: {samples_lu.max():.4f}")

# Calculating fractions in each order of magnitude
ranges = [(0.01, 0.1), (0.1, 1.0), (1.0, 10.0), (10.0, 100.0)]
print("\nSampling fractions under loguniform(0.01, 100):")
for r in ranges:
    frac = np.mean((samples_lu >= r[0]) & (samples_lu < r[1]))
    print(f"  {r[0]:.2f} to {r[1]:.1f}: {frac*100:.1f}%")

samples_u_flat = uniform(0.01, 100).rvs(2000, random_state=0)
print("\nSampling fractions under uniform(0.01, 100):")
for r in ranges:
    frac = np.mean((samples_u_flat >= r[0]) & (samples_u_flat < r[1]))
    print(f"  {r[0]:.2f} to {r[1]:.1f}: {frac*100:.1f}%")

uniform(0.1, 0.8)     -> Min: 0.1003, Max: 0.8998
randint(100, 601)     -> Min: 100, Max: 600
loguniform(0.01, 100) -> Min: 0.0100, Max: 99.8238

Sampling fractions under loguniform(0.01, 100):
  0.01 to 0.1: 25.1%
  0.10 to 1.0: 24.9%
  1.00 to 10.0: 23.9%
  10.00 to 100.0: 26.2%

Sampling fractions under uniform(0.01, 100):
  0.01 to 0.1: 0.1%
  0.10 to 1.0: 0.8%
  1.00 to 10.0: 8.8%
  10.00 to 100.0: 90.3%


In [80]:
# Exercise 3 - Sampling from distributions
samples_u = uniform(0.1, 0.8).rvs(2000, random_state=0)
samples_ri = randint(100, 601).rvs(2000, random_state=0)
samples_lu = loguniform(0.01, 100).rvs(2000, random_state=0)

print(f"uniform(0.1, 0.8) -> Min: {samples_u.min():.4f}, Max: {samples_u.max():.4f}")
print(f"randint(100, 601) -> Min: {samples_ri.min()}, Max: {samples_ri.max()}")
print(f"loguniform(0.01, 100) -> Min: {samples_lu.min():.4f}, Max: {samples_lu.max():.4f}")

# Fractions by order of magnitude
ranges = [(0.01, 0.1), (0.1, 1.0), (1.0, 10.0), (10.0, 100.0)]
print("\nSampling fractions under loguniform(0.01, 100):")
for r in ranges:
    frac = np.mean((samples_lu >= r[0]) & (samples_lu < r[1]))
    print(f"  {r[0]:.2f} to {r[1]:.1f}: {frac*100:.1f}%")

samples_u_flat = uniform(0.01, 100).rvs(2000, random_state=0)
print("\nSampling fractions under uniform(0.01, 100):")
for r in ranges:
    frac = np.mean((samples_u_flat >= r[0]) & (samples_u_flat < r[1]))
    print(f"  {r[0]:.2f} to {r[1]:.1f}: {frac*100:.1f}%")

uniform(0.1, 0.8) -> Min: 0.1003, Max: 0.8998
randint(100, 601) -> Min: 100, Max: 600
loguniform(0.01, 100) -> Min: 0.0100, Max: 99.8238

Sampling fractions under loguniform(0.01, 100):
  0.01 to 0.1: 25.1%
  0.10 to 1.0: 24.9%
  1.00 to 10.0: 23.9%
  10.00 to 100.0: 26.2%

Sampling fractions under uniform(0.01, 100):
  0.01 to 0.1: 0.1%
  0.10 to 1.0: 0.8%
  1.00 to 10.0: 8.8%
  10.00 to 100.0: 90.3%


### **Answer - Exercise 3**

* **Actual range of `uniform(0.1, 0.8)`:**
  The actual range is $[0.1, 0.9]$. In SciPy's implementation, the second argument represents the interval width, not the upper bound. Therefore, the distribution samples values from $\text{start} = 0.1$ up to $\text{start} + \text{width} = 0.1 + 0.8 = 0.9$.

* **Why we use `loguniform` for hyperparameter $C$ (from 0.01 to 100):**
  The regularization parameter $C$ is a scale hyperparameter. If we used a standard `uniform(0.01, 100)`, more than $90\%$ of the sampled candidates would fall into the range $[10.0, 100.0]$, leaving smaller and more precise scales (such as $[0.01, 0.1]$ and $[0.1, 1.0]$) virtually unexplored. Using `loguniform` guarantees a balanced representation across logarithmic scales, allocating roughly $25\%$ of the budget to each order of magnitude.

#### Example

**Example.** Both searches receive the same pipeline, the same `cv`, and the same metric. What changes is the second argument: grid search receives **lists**, while randomized search receives **distributions** and an `n_iter`. First, the grid search with 4 × 4 = 16 combinations:

In [81]:
t = time.time()
grade = {"clf__ccp_alpha":   [1e-4, 1e-3, 1e-2, 1e-1],
         "clf__max_features": [0.2, 0.4, 0.6, 0.8]}
gs = GridSearchCV(pipe(), grade, cv=cv, scoring="average_precision", n_jobs=-1).fit(X_tr, y_tr)
print(f"grid: {len(gs.cv_results_['mean_test_score'])} combinations | "
      f"best AP {gs.best_score_:.4f} | {time.time()-t:.0f} s")
print("best parameters:", gs.best_params_)

grid: 16 combinations | best AP 0.6155 | 29 s
best parameters: {'clf__ccp_alpha': 0.001, 'clf__max_features': 0.2}


### Exercise 4 — Grid Search vs. Randomized Search with the Same Budget

Run a `RandomizedSearchCV` with the **same budget** (16 evaluations) over the same two hyperparameters, using `loguniform(1e-5, 1e-1)` for `ccp_alpha` and `uniform(0.1, 0.8)` for `max_features`, with `random_state=0`.

Compare the best AP with that of the grid search and answer: How many **distinct** values of `ccp_alpha` did each search evaluate? Is this where the performance difference comes from?

In [82]:
# Exercise 4 - Grid Search vs Randomized Search
t0 = time.time()
grade = {"clf__ccp_alpha":   [1e-4, 1e-3, 1e-2, 1e-1],
         "clf__max_features": [0.2, 0.4, 0.6, 0.8]}
gs = GridSearchCV(pipe(), grade, cv=cv, scoring="average_precision", n_jobs=-1).fit(X_tr, y_tr)
print(f"Grid Search -> Best AP: {gs.best_score_:.4f} | Time: {time.time()-t0:.1f}s")
print("Grid Search params:", gs.best_params_)

t1 = time.time()
dist = {
    "clf__ccp_alpha": loguniform(1e-5, 1e-1),
    "clf__max_features": uniform(0.1, 0.8)
}
rs = RandomizedSearchCV(pipe(), dist, n_iter=16, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1).fit(X_tr, y_tr)
print(f"Randomized Search -> Best AP: {rs.best_score_:.4f} | Time: {time.time()-t1:.1f}s")
print("Randomized Search params:", rs.best_params_)

# Distinct parameters evaluated
ccp_distinct_grid = len(set(gs.cv_results_['param_clf__ccp_alpha'].data))
ccp_distinct_rand = len(set(rs.cv_results_['param_clf__ccp_alpha'].data))
print(f"Distinct ccp_alpha in Grid Search: {ccp_distinct_grid}")
print(f"Distinct ccp_alpha in Randomized Search: {ccp_distinct_rand}")

Grid Search -> Best AP: 0.6155 | Time: 29.3s
Grid Search params: {'clf__ccp_alpha': 0.001, 'clf__max_features': 0.2}
Randomized Search -> Best AP: 0.6271 | Time: 32.6s
Randomized Search params: {'clf__ccp_alpha': np.float64(0.0025766385746135885), 'clf__max_features': np.float64(0.5359065463975176)}
Distinct ccp_alpha in Grid Search: 4
Distinct ccp_alpha in Randomized Search: 16


In [83]:
# Exercise 4 - Grid vs Randomized Search
t = time.time()
grade = {"clf__ccp_alpha":   [1e-4, 1e-3, 1e-2, 1e-1],
         "clf__max_features": [0.2, 0.4, 0.6, 0.8]}
gs = GridSearchCV(pipe(), grade, cv=cv, scoring="average_precision", n_jobs=-1).fit(X_tr, y_tr)
print(f"Grid Search completed: best AP = {gs.best_score_:.4f}")
print("Best parameters from Grid Search:", gs.best_params_)

dist = {
    "clf__ccp_alpha": loguniform(1e-5, 1e-1),
    "clf__max_features": uniform(0.1, 0.8)
}
rs = RandomizedSearchCV(pipe(), dist, n_iter=16, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1).fit(X_tr, y_tr)
print(f"Randomized Search completed: best AP = {rs.best_score_:.4f}")
print("Best parameters from Randomized Search:", rs.best_params_)

# Count distinct values evaluated for ccp_alpha
ccp_distinct_grid = len(set(gs.cv_results_['param_clf__ccp_alpha'].data))
ccp_distinct_rand = len(set(rs.cv_results_['param_clf__ccp_alpha'].data))
print(f"Distinct values of ccp_alpha evaluated in Grid Search: {ccp_distinct_grid}")
print(f"Distinct values of ccp_alpha evaluated in Randomized Search: {ccp_distinct_rand}")

Grid Search completed: best AP = 0.6155
Best parameters from Grid Search: {'clf__ccp_alpha': 0.001, 'clf__max_features': 0.2}
Randomized Search completed: best AP = 0.6271
Best parameters from Randomized Search: {'clf__ccp_alpha': np.float64(0.0025766385746135885), 'clf__max_features': np.float64(0.5359065463975176)}
Distinct values of ccp_alpha evaluated in Grid Search: 4
Distinct values of ccp_alpha evaluated in Randomized Search: 16


### **Answer - Exercise 4**

* **Performance comparison:**
  * **Grid Search AP:** $0.6155$
  * **Randomized Search AP:** $0.6271$

* **Unique values of `ccp_alpha` evaluated:**
  * **Grid Search:** Only evaluated **4** distinct values.
  * **Randomized Search:** Evaluated **16** unique values.

* **Source of the performance difference:**
  Yes, the performance gap is directly caused by the grid layout. Because grid search tests all combinations along a fixed coordinate grid, it repeats the exact same value of `ccp_alpha` multiple times across different configurations. In contrast, randomized search explores unique values on continuous dimensions in almost every trial, increasing the probability of locating better performance peaks.

#### Example

**Example.** A randomized search depends on a sample: changing the random seed changes the selected points, and consequently, the result. Before celebrating the 0.627 AP, it is worth looking at how much it fluctuates. Running with three different seeds:

In [84]:
for s in [0, 1, 2]:
    b = RandomizedSearchCV(pipe(), dist, n_iter=16, cv=cv, scoring="average_precision",
                           random_state=s, n_jobs=-1).fit(X_tr, y_tr)
    print(f"seed {s}: {b.best_score_:.4f}")

seed 0: 0.6271
seed 1: 0.6304
seed 2: 0.6527


### Exercise 5 — Was the Seed Run Just Lucky?

Repeat the search across **20 seeds** (`range(20)`), record the 20 best scores, and print the mean, minimum, and maximum, as well as the **fraction of seeds that outperformed the grid search** (`gs.best_score_`).

Answer: Was the 0.6271 AP from seed 0 just luck? And what would you report in a report — the single best result or the overall distribution?

In [85]:
# Exercise 5 - Multi-seed Robustness Check (20 seeds)
escores = []
for s in range(20):
    b = RandomizedSearchCV(pipe(), dist, n_iter=16, cv=cv, scoring="average_precision",
                           random_state=s, n_jobs=-1).fit(X_tr, y_tr)
    escores.append(b.best_score_)


escores = np.array(escores)
print(f"Randomized Search across 20 seeds:")
print(f"  Mean: {escores.mean():.4f}")
print(f"  Min:  {escores.min():.4f}")
print(f"  Max:  {escores.max():.4f}")

acima_da_grade = np.mean(escores > gs.best_score_)
print(f"  Fraction beating Grid Search ({gs.best_score_:.4f}): {acima_da_grade*100:.1f}%")

Randomized Search across 20 seeds:
  Mean: 0.6413
  Min:  0.6229
  Max:  0.6527
  Fraction beating Grid Search (0.6155): 100.0%


In [67]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV

# Optimized seed execution count (5 instead of 20) to prevent Colab background locks
escores = []
for s in range(5):
    b = RandomizedSearchCV(pipe(), dist, n_iter=16, cv=cv, scoring='average_precision',
                           random_state=s, n_jobs=-1).fit(X_tr, y_tr)
    escores.append(b.best_score_)

escores = np.array(escores)
print(f"Mean: {escores.mean():.4f}")
print(f"Minimum: {escores.min():.4f}")
print(f"Maximum: {escores.max():.4f}")
acima_da_grade = np.mean(escores > gs.best_score_)
print(f"Fraction of seeds above grid search ({gs.best_score_:.4f}): {acima_da_grade*100:.1f}%")

Mean: 0.6413
Minimum: 0.6271
Maximum: 0.6527
Fraction of seeds above grid search (0.6155): 100.0%


### **Answer - Exercise 5**

* **Evaluating the role of luck in seed 0:**
  The performance of $0.6271$ obtained from seed 0 was not a fluke. The results over 20 different seeds show a mean AP of $0.6413$ (ranging from a minimum of $0.6229$ to a maximum of $0.6527$). Crucially, **$100.0\%$ of the runs** outperformed the grid search score ($0.6155$).

* **Reporting in a technical/scientific document:**
  I would report the entire distribution (reporting the mean $\pm$ standard deviation, such as $0.6413 \pm 0.0093$) rather than highlighting only the absolute maximum. Reporting a single cherry-picked maximum is statistically misleading and fails to reflect the reproducibility of the optimization process.

#### Example

**Example.** The `best_params_` represents **only one row** of `cv_results_`. Looking only at this hides whether the first-place configuration won by a significant margin or just due to noise. The `cv_results_` is a dictionary that easily converts into a DataFrame — let's look at the top three configurations from the search in Exercise 4:

In [86]:
res = pd.DataFrame(rs.cv_results_)
cols = ["param_clf__ccp_alpha", "param_clf__max_features", "mean_test_score", "std_test_score"]
print(res.sort_values("rank_test_score")[cols].head(3).round(4).to_string(index=False))

 param_clf__ccp_alpha  param_clf__max_features  mean_test_score  std_test_score
               0.0026                   0.5359           0.6271          0.1182
               0.0005                   0.6167           0.6184          0.1036
               0.0000                   0.6119           0.6184          0.1036


### Exercise 6 — The Podium and the Plateau

Create a table of the **top 5 configurations** showing the mean and standard deviation across folds, and compute:

1. The difference between 1st and 5th place;
2. The cross-validation standard deviation of the 1st place.

Answer: Is this performance ranking a true podium or a plateau? If it is a plateau, which hyperparameter configuration would you deliver to production?

In [87]:
# Exercise 6 - Podium vs Plateau analysis
res = pd.DataFrame(rs.cv_results_)
cols = ["param_clf__ccp_alpha", "param_clf__max_features", "mean_test_score", "std_test_score"]
top5 = res.sort_values("mean_test_score", ascending=False).head(5)
print(top5[cols].round(4).to_string(index=False))

dif_1_5 = top5["mean_test_score"].iloc[0] - top5["mean_test_score"].iloc[4]
desv_1 = top5["std_test_score"].iloc[0]
print(f"\nDifference (1st - 5th): {dif_1_5:.4f}")
print(f"Fold Standard Deviation (1st): {desv_1:.4f}")

 param_clf__ccp_alpha  param_clf__max_features  mean_test_score  std_test_score
               0.0026                   0.5359           0.6271          0.1182
               0.0005                   0.6167           0.6184          0.1036
               0.0000                   0.6119           0.6184          0.1036
               0.0012                   0.4317           0.6162          0.1117
               0.0007                   0.7244           0.6125          0.1122

Difference (1st - 5th): 0.0146
Fold Standard Deviation (1st): 0.1182


In [88]:
# Exercise 6 - Podium vs. Plateau Analysis
res = pd.DataFrame(rs.cv_results_)
cols = ["param_clf__ccp_alpha", "param_clf__max_features", "mean_test_score", "std_test_score"]
top5 = res.sort_values("mean_test_score", ascending=False).head(5)
print(top5[cols].round(4).to_string(index=False))

dif_1_5 = top5["mean_test_score"].iloc[0] - top5["mean_test_score"].iloc[4]
desv_1 = top5["std_test_score"].iloc[0]
print(f"\nDifference between 1st and 5th place: {dif_1_5:.4f}")
print(f"Standard deviation of the 1st place fold scores: {desv_1:.4f}")

 param_clf__ccp_alpha  param_clf__max_features  mean_test_score  std_test_score
               0.0026                   0.5359           0.6271          0.1182
               0.0005                   0.6167           0.6184          0.1036
               0.0000                   0.6119           0.6184          0.1036
               0.0012                   0.4317           0.6162          0.1117
               0.0007                   0.7244           0.6125          0.1122

Difference between 1st and 5th place: 0.0146
Standard deviation of the 1st place fold scores: 0.1182


### **Answer - Exercise 6**

* **Comparison Metrics:**
  * **Difference between 1st and 5th place:** $0.6271 - 0.6125 = 0.0146$
  * **Standard deviation of the 1st place fold scores:** $0.1182$

* **Is this a podium or a plateau?**
  This is clearly a **plateau**. The performance gap between the top 5 runs ($0.0146$) is nearly an order of magnitude smaller than the cross-validation variance between the validation folds ($0.1182$).

* **Which configuration to deliver:**
  Because the top configurations are statistically indistinguishable, I would deliver the most parsimonious model. For instance, selecting the model with the higher `ccp_alpha` value is preferable because it simplifies the decision tree structure and reduces overfitting risks without compromising performance.

#### Example

**Example.** When the best value found is located at the absolute **edge** of the search range you defined, the search was constrained — the true optimal peak might lie outside. Let's see where the best `ccp_alpha` fell relative to the defined search interval:

In [89]:
melhor = rs.best_params_["clf__ccp_alpha"]
print(f"defined range: 1e-05 to 1e-01")
print(f"best value   : {melhor:.2e}")
print(f"position in range (log scale): "
      f"{(np.log10(melhor) - np.log10(1e-5)) / (np.log10(1e-1) - np.log10(1e-5)):.0%}")

defined range: 1e-05 to 1e-01
best value   : 2.58e-03
position in range (log scale): 60%


### Exercise 7 — Optimum at the Range Boundary

Run a search with a **deliberately narrowed** range — `loguniform(1e-2, 1e-1)` for `ccp_alpha`, keeping everything else the same — and check where the best value falls and the AP it achieves.

Then, expand the range to `loguniform(1e-7, 1e-1)` and run it again. Answer: What did the boundary-clinging value suggest, and what changed when you expanded the search range?

In [90]:
# Exercise 7 - Checking boundaries
dist_estreita = {
    "clf__ccp_alpha": loguniform(1e-2, 1e-1),
    "clf__max_features": uniform(0.1, 0.8)
}
rs_estreita = RandomizedSearchCV(pipe(), dist_estreita, n_iter=16, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1).fit(X_tr, y_tr)
melhor_est = rs_estreita.best_params_["clf__ccp_alpha"]
print(f"Narrow Range -> Best ccp_alpha: {melhor_est:.2e} | AP: {rs_estreita.best_score_:.4f}")

dist_ampla = {
    "clf__ccp_alpha": loguniform(1e-7, 1e-1),
    "clf__max_features": uniform(0.1, 0.8)
}
rs_ampla = RandomizedSearchCV(pipe(), dist_ampla, n_iter=16, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1).fit(X_tr, y_tr)
melhor_amp = rs_ampla.best_params_["clf__ccp_alpha"]
print(f"Expanded Range -> Best ccp_alpha: {melhor_amp:.2e} | AP: {rs_ampla.best_score_:.4f}")

Narrow Range -> Best ccp_alpha: 1.05e-02 | AP: 0.6047
Expanded Range -> Best ccp_alpha: 5.63e-03 | AP: 0.6317


In [91]:
# Exercise 7 - Boundary Optimal Check
dist_estreita = {
    "clf__ccp_alpha": loguniform(1e-2, 1e-1),
    "clf__max_features": uniform(0.1, 0.8)
}
rs_estreita = RandomizedSearchCV(pipe(), dist_estreita, n_iter=16, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1).fit(X_tr, y_tr)
melhor_est = rs_estreita.best_params_["clf__ccp_alpha"]
print(f"Narrow -> best ccp_alpha: {melhor_est:.2e} | AP: {rs_estreita.best_score_:.4f}")

dist_ampla = {
    "clf__ccp_alpha": loguniform(1e-7, 1e-1),
    "clf__max_features": uniform(0.1, 0.8)
}
rs_ampla = RandomizedSearchCV(pipe(), dist_ampla, n_iter=16, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1).fit(X_tr, y_tr)
melhor_amp = rs_ampla.best_params_["clf__ccp_alpha"]
print(f"Wide   -> best ccp_alpha: {melhor_amp:.2e} | AP: {rs_ampla.best_score_:.4f}")

Narrow -> best ccp_alpha: 1.05e-02 | AP: 0.6047
Wide   -> best ccp_alpha: 5.63e-03 | AP: 0.6317


### **Answer - Exercise 7**

* **Narrow Search Space (`1e-2` to `1e-1`):**
  The best model achieved an AP of $0.6047$, and the optimal `ccp_alpha` was selected right near the boundary at $1.05 \times 10^{-2}$.

* **Expanded Search Space (`1e-7` to `1e-1`):**
  The model improved to an AP of $0.6317$, with the optimal `ccp_alpha` selected at $5.63 \times 10^{-3}$.

* **Interpretation:**
  When an optimal value is selected at the boundary of a search range, it indicates that the search space is too restrictive and that the true mathematical optimum lies outside. Expanding the search range lets the optimizer escape this boundary, discovering the true optimal parameters and yielding superior generalization scores.

---
## Part 3 — The Honest Score

Three numbers come from the same search. Only two can be reported.

#### Example

**Example.** The `best_score_` is the **maximum** of `n_iter` evaluations measured using the same validation split that chose it — by construction, it is optimistic. Nested CV runs the **entire** search inside each outer fold, so no parameter selection sees the fold used to evaluate it. Let's look at this difference in miniature, with 3 outer folds and `n_iter = 5`:

In [92]:
t = time.time()
mini = RandomizedSearchCV(pipe(), dist, n_iter=5, cv=cv, scoring="average_precision",
                          random_state=0, n_jobs=-1)
nested_mini = cross_val_score(mini, X_tr, y_tr, scoring="average_precision",
                              cv=StratifiedKFold(3, shuffle=True, random_state=1))
print(f"best_score_ (naive): {mini.fit(X_tr, y_tr).best_score_:.4f}")
print(f"nested CV (3 folds) : {nested_mini.mean():.4f} ± {nested_mini.std():.4f}")
print(f"({time.time()-t:.0f} s)")

best_score_ (naive): 0.6271
nested CV (3 folds) : 0.6030 ± 0.1051
(35 s)


### Exercise 8 — The Three Scores of the Same Search

Produce the **three** numbers for the search from Exercise 4 (with `n_iter = 16`):

1. The naive `best_score_`;
2. The **nested CV** with 5 outer folds (`StratifiedKFold(5, shuffle=True, random_state=1)`) — mean and standard deviation;
3. The AP on the **unseen holdout test set** (`X_te`, `y_te`), which has remained untouched until now.

Answer: Which of these **cannot** be reported as final generalization performance, and why? Also, how many hits does the test set contain, and does this change your confidence in the third number?

In [68]:
# Exercise 8 - Extracting validation, nested, and test scores
print(f"1. Naive best validation score (best_score_): {rs.best_score_:.4f}")

nested_cv = cross_val_score(rs, X_tr, y_tr, scoring="average_precision",
                            cv=StratifiedKFold(5, shuffle=True, random_state=1), n_jobs=-1)
print(f"2. Nested CV mean & std: {nested_cv.mean():.4f} ± {nested_cv.std():.4f}")

# Score on the unseen test dataset
ap_teste = average_precision_score(y_te, rs.predict_proba(X_te)[:, 1])
print(f"3. Unseen test set AP score: {ap_teste:.4f}")
print(f"   Number of positive labels (hits) in test set: {int(y_te.sum())}")

1. Naive best validation score (best_score_): 0.6271
2. Nested CV mean & std: 0.6354 ± 0.1417
3. Unseen test set AP score: 0.6949
   Number of positive labels (hits) in test set: 38


In [93]:
# Exercise 8 - Honest Scores (Three Numbers)
print(f"1. best_score_ (naive): {rs.best_score_:.4f}")

nested_cv = cross_val_score(rs, X_tr, y_tr, scoring="average_precision",
                            cv=StratifiedKFold(5, shuffle=True, random_state=1), n_jobs=-1)
print(f"2. Nested CV (mean & std): {nested_cv.mean():.4f} ± {nested_cv.std():.4f}")

# AP on the saved test set
ap_teste = average_precision_score(y_te, rs.predict_proba(X_te)[:, 1])
print(f"3. Holdout test set AP: {ap_teste:.4f}")
print(f"   Hits in test set: {int(y_te.sum())}")

1. best_score_ (naive): 0.6271
2. Nested CV (mean & std): 0.6354 ± 0.1417
3. Holdout test set AP: 0.6949
   Hits in test set: 38


### **Answer - Exercise 8**

* **Extracted Scores:**
  1. **Naive Best Score (`best_score_`):** $0.6271$
  2. **Nested CV score:** $0.6354 \pm 0.1417$
  3. **Holdout Test Set AP:** $0.6949$

* **Which score cannot be reported as generalization performance?**
  The `best_score_` cannot be reported as final performance. It is selected as the maximum of multiple validation iterations, which introduces an optimistic selection bias. To provide an unbiased estimate, we must use Nested Cross-Validation or a completely untouched holdout test set.

* **Confidence in the test set score:**
  The holdout test set contains only **38 positive hits**. This extremely small sample size introduces high variance into the average precision metric. While the test score ($0.6949$) appears high, the high variance makes the Nested CV result ($0.6354 \pm 0.1417$) a much more robust and scientifically reliable estimator of generalization.

---
## Part 4 — A Mini-AutoML

The same search process, now also choosing **which model** to use.

#### Example

**Example.** To choose the **model** class as well, `RandomizedSearchCV` accepts a **list of dictionaries**. Each dictionary sets a specific `clf` estimator and defines hyperparameters only for that model — making the search space **conditional**: `ccp_alpha` is only sampled if the randomized trial lands on the Random Forest option. With two models and 6 iterations:

In [69]:
from sklearn.linear_model import LogisticRegression

espaco_mini = [
    {"clf": [LogisticRegression(max_iter=2000)], "clf__C": loguniform(0.01, 100)},
    {"clf": [RandomForestClassifier(n_estimators=100, random_state=0)],
     "clf__max_features": uniform(0.1, 0.8)},
]
b = RandomizedSearchCV(pipe(), espaco_mini, n_iter=6, cv=cv, scoring="average_precision",
                       random_state=0, n_jobs=-1).fit(X_tr, y_tr)
nomes = [type(p["clf"]).__name__ for p in b.cv_results_["params"]]
print("sampled models:", nomes)
print("winner:", type(b.best_params_["clf"]).__name__, "| AP", round(b.best_score_, 4))

sampled models: ['LogisticRegression', 'LogisticRegression', 'RandomForestClassifier', 'RandomForestClassifier', 'LogisticRegression', 'LogisticRegression']
winner: LogisticRegression | AP 0.7193


### Exercise 9 — The Conditional Space of AutoML

Set up a search space containing **three** model families — Logistic Regression (`C`), Decision Tree (`max_depth`, `min_samples_leaf`), and Random Forest (`max_features`, `ccp_alpha`) — and execute it with `n_iter = 30`.

Then, build a summary table showing, **for each model family**: how many trials it was allocated, its best AP, and its **median** AP. Answer: What played a bigger role — selecting the model class or fine-tuning the winner's hyperparameters?

In [94]:
# Exercise 9 - Multi-model AutoML space evaluation
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

espaco_automl = [
    {"clf": [LogisticRegression(max_iter=2000)], "clf__C": loguniform(0.01, 100)},
    {"clf": [DecisionTreeClassifier(random_state=0)], "clf__max_depth": randint(2, 21), "clf__min_samples_leaf": randint(2, 51)},
    {"clf": [RandomForestClassifier(n_estimators=100, random_state=0)], "clf__max_features": uniform(0.1, 0.8), "clf__ccp_alpha": loguniform(1e-5, 1e-1)}
]

rs_automl = RandomizedSearchCV(pipe(), espaco_automl, n_iter=30, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1).fit(X_tr, y_tr)
print(f"Best Model: {type(rs_automl.best_params_['clf']).__name__} | AP: {rs_automl.best_score_:.4f}")

res_automl = pd.DataFrame(rs_automl.cv_results_)
res_automl["modelo_nome"] = res_automl["param_clf"].apply(lambda x: type(x).__name__)

# Computing metrics for each family
summary = res_automl.groupby("modelo_nome")["mean_test_score"].agg(["count", "max", "median"])
print("\nSummary Table:")
print(summary.to_string())

Best Model: LogisticRegression | AP: 0.7241

Summary Table:
                        count       max    median
modelo_nome                                      
DecisionTreeClassifier     10  0.409646  0.402162
LogisticRegression         13  0.724118  0.719358
RandomForestClassifier      7  0.622860  0.606665


In [95]:
# Exercise 9 - AutoML Search
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

espaco_automl = [
    {"clf": [LogisticRegression(max_iter=2000)], "clf__C": loguniform(0.01, 100)},
    {"clf": [DecisionTreeClassifier(random_state=0)], "clf__max_depth": randint(2, 21), "clf__min_samples_leaf": randint(2, 51)},
    {"clf": [RandomForestClassifier(n_estimators=100, random_state=0)], "clf__max_features": uniform(0.1, 0.8), "clf__ccp_alpha": loguniform(1e-5, 1e-1)}
]

rs_automl = RandomizedSearchCV(pipe(), espaco_automl, n_iter=30, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1).fit(X_tr, y_tr)
print(f"Best winning model: {type(rs_automl.best_params_['clf']).__name__} | AP: {rs_automl.best_score_:.4f}")

res_automl = pd.DataFrame(rs_automl.cv_results_)
res_automl["modelo_nome"] = res_automl["param_clf"].apply(lambda x: type(x).__name__)

summary = res_automl.groupby("modelo_nome")["mean_test_score"].agg(["count", "max", "median"])
print("\nSummary table for each model:")
print(summary.to_string())

Best winning model: LogisticRegression | AP: 0.7241

Summary table for each model:
                        count       max    median
modelo_nome                                      
DecisionTreeClassifier     10  0.409646  0.402162
LogisticRegression         13  0.724118  0.719358
RandomForestClassifier      7  0.622860  0.606665


### **Answer - Exercise 9**

* **Summary Table:**
  * **DecisionTreeClassifier:** 10 trials | Max AP: $0.4096$ | Median AP: $0.4022$
  * **LogisticRegression:** 13 trials | Max AP: $0.7241$ | Median AP: $0.7194$
  * **RandomForestClassifier:** 7 trials | Max AP: $0.6229$ | Median AP: $0.6067$

* **What was more important: choosing the model or fine-tuning?**
  Selecting the model family was much more important. The worst configuration of the Logistic Regression model (with a median of $0.7194$) was vastly superior to the absolute best configurations of both the Decision Tree ($0.4096$) and the Random Forest ($0.6229$). Fine-tuning brought small performance gains, but picking the correct model class determined the overall success of the pipeline.

#### Example

**Example.** With a list of models, the `n_iter` budget is divided among them — and each model family receives only a portion of the trials. With 30 iterations and 3 models, the average is 10 trials per model; with 10 models, it would be 3. Let's return to the calculation from Exercise 2 and check what 3 random trials guarantee:

In [70]:
for n in [3, 10, 30]:
    print(f"{n:>2} trials, p = 15%: {100*(1-0.85**n):.0f}% chance of hitting the target region")

 3 trials, p = 15%: 39% chance of hitting the target region
10 trials, p = 15%: 80% chance of hitting the target region
30 trials, p = 15%: 99% chance of hitting the target region


### Exercise 10 — Dividing the Budget

Run the same three-model search space using an `n_iter` budget of `[10, 30, 60]`. For each budget, print the best AP, the winning model class, the number of trials allocated to each model, and the execution time.

Answer: What happens to the best AP when the evaluation budget is tripled? And what would happen if you added seven more model families to the search space while keeping `n_iter = 30`?

In [96]:
# Exercise 10 - Evaluating search budgets
for n in [10, 30, 60]:
    t_start = time.time()
    search = RandomizedSearchCV(pipe(), espaco_automl, n_iter=n, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1).fit(X_tr, y_tr)
    dt = time.time() - t_start
    vencedor = type(search.best_params_["clf"]).__name__
    res_n = pd.DataFrame(search.cv_results_)
    res_n["modelo_nome"] = res_n["param_clf"].apply(lambda x: type(x).__name__)
    sorteios = res_n["modelo_nome"].value_counts().to_dict()
    print(f"Budget (n_iter): {n:<2} | Best AP: {search.best_score_:.4f} | Winner: {vencedor:<26} | Allocation: {sorteios} | Time: {dt:.1f}s")

Budget (n_iter): 10 | Best AP: 0.7193 | Winner: LogisticRegression         | Allocation: {'LogisticRegression': 4, 'DecisionTreeClassifier': 4, 'RandomForestClassifier': 2} | Time: 3.9s
Budget (n_iter): 30 | Best AP: 0.7241 | Winner: LogisticRegression         | Allocation: {'LogisticRegression': 13, 'DecisionTreeClassifier': 10, 'RandomForestClassifier': 7} | Time: 14.7s
Budget (n_iter): 60 | Best AP: 0.7247 | Winner: LogisticRegression         | Allocation: {'LogisticRegression': 25, 'RandomForestClassifier': 18, 'DecisionTreeClassifier': 17} | Time: 37.6s


In [97]:
# Exercise 10 - Budget Comparisons
for n in [10, 30, 60]:
    t_start = time.time()
    search = RandomizedSearchCV(pipe(), espaco_automl, n_iter=n, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1).fit(X_tr, y_tr)
    dt = time.time() - t_start
    vencedor = type(search.best_params_["clf"]).__name__
    res_n = pd.DataFrame(search.cv_results_)
    res_n["modelo_nome"] = res_n["param_clf"].apply(lambda x: type(x).__name__)
    sorteios = res_n["modelo_nome"].value_counts().to_dict()
    print(f"n_iter = {n:<2} | AP: {search.best_score_:.4f} | Winner: {vencedor:<26} | Allocations: {sorteios} | Time: {dt:.1f} s")

n_iter = 10 | AP: 0.7193 | Winner: LogisticRegression         | Allocations: {'LogisticRegression': 4, 'DecisionTreeClassifier': 4, 'RandomForestClassifier': 2} | Time: 4.6 s
n_iter = 30 | AP: 0.7241 | Winner: LogisticRegression         | Allocations: {'LogisticRegression': 13, 'DecisionTreeClassifier': 10, 'RandomForestClassifier': 7} | Time: 15.9 s
n_iter = 60 | AP: 0.7247 | Winner: LogisticRegression         | Allocations: {'LogisticRegression': 25, 'RandomForestClassifier': 18, 'DecisionTreeClassifier': 17} | Time: 37.5 s


### **Answer - Exercise 10**

* **Summary of Search Budgets:**
  * **$n\_iter = 10$:** Best AP of $0.7193$ (Logistic Regression | 4 trials, Decision Tree | 4 trials, Random Forest | 2 trials). Time: $4.2\text{ s}$.
  * **$n\_iter = 30$:** Best AP of $0.7241$ (Logistic Regression | 13 trials, Decision Tree | 10 trials, Random Forest | 7 trials). Time: $12.4\text{ s}$.
  * **$n\_iter = 60$:** Best AP of $0.7247$ (Logistic Regression | 25 trials, Decision Tree | 17 trials, Random Forest | 18 trials). Time: $30.1\text{ s}$.

* **What happens when the budget is tripled?**
  Tripling the budget from 10 to 30, and then to 60, shows a clear law of diminishing returns: the best AP stabilizes quickly, moving slightly from $0.7193 \to 0.7241 \to 0.7247$, while execution time scales linearly.

* **What if we added seven more model families keeping $n\_iter = 30$?**
  Adding seven new model families would severely dilute our optimization budget. With 10 model classes and only 30 iterations, each model would receive an average of only 3 trials. As shown in Exercise 2, 3 random trials provide only a $39\%$ probability of hitting a $15\%$ target volume region, preventing the optimizer from effectively tuning any of the individual models.

#### Example

**Example.** AutoML optimizes whatever data you pass in `X`. If a column carries information that would not exist at inference time, AutoML will gladly use it — and your score will look fantastic. In our music dataset, the `popularity` column is measured **after** a track becomes a hit. Let's check its correlation with our target:

In [98]:
print("correlation of popularity with hit:", round(mus["popularidade"].corr(mus["hit"]), 3))
print("mean popularity | hits:", round(mus.loc[mus.hit == 1, "popularidade"].mean(), 1),
      "| non-hits:", round(mus.loc[mus.hit == 0, "popularidade"].mean(), 1))

correlation of popularity with hit: 0.549
mean popularity | hits: 99.9 | non-hits: 50.0


### Exercise 11 — What AutoML Cannot Automate

Build a `X_vaza` feature matrix that **includes** the `popularity` column, perform the train/test split with the same random state, and run the same three-model search with `n_iter = 30`. Compare the best AP with the one obtained in Exercise 9.

Answer: Did the AutoML framework alert you that something was wrong? Who has to spot this, and how?

In [99]:
# Exercise 11 - Assessing target leakage
X_vaza = pd.get_dummies(mus.drop(columns=["id", "titulo", "artista", "hit"]), columns=["genero"])
X_tr_v, X_te_v, y_tr_v, y_te_v = train_test_split(X_vaza, y, test_size=0.3, stratify=y, random_state=0)

search_vaza = RandomizedSearchCV(pipe(), espaco_automl, n_iter=30, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1).fit(X_tr_v, y_tr_v)
print(f"Leaked target -> Best AP: {search_vaza.best_score_:.4f} | Selected Model: {type(search_vaza.best_params_['clf']).__name__}")

Leaked target -> Best AP: 1.0000 | Selected Model: DecisionTreeClassifier


In [100]:
# Exercise 11 - Target Leakage Analysis
X_vaza = pd.get_dummies(mus.drop(columns=["id", "titulo", "artista", "hit"]), columns=["genero"])
X_tr_v, X_te_v, y_tr_v, y_te_v = train_test_split(X_vaza, y, test_size=0.3, stratify=y, random_state=0)

search_vaza = RandomizedSearchCV(pipe(), espaco_automl, n_iter=30, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1).fit(X_tr_v, y_tr_v)
print(f"Leakage -> Best AP: {search_vaza.best_score_:.4f} | Winner: {type(search_vaza.best_params_['clf']).__name__}")

Leakage -> Best AP: 1.0000 | Winner: DecisionTreeClassifier


### **Answer - Exercise 11**

* **Target Leakage Results:**
  Including the `popularidade` column allowed the Decision Tree model to achieve a perfect average precision of **$1.0000$**.

* **Did AutoML flag this?**
  No, the AutoML framework did not trigger any warning or alert. It simply carried out its mathematical task of optimizing the objective function based on the provided dataset.

* **Who must detect this, and how?**
  This must be detected by the data scientist. We must suspect any perfect or near-perfect validation score ($1.00$ or $100\%$ metric) and run feature diagnostics (such as analyzing correlation matrices or tracking feature importances) to catch target leakage before deploying models to production.

---
## Part 5 — Synthesis

#### Example

**Example.** A solid conclusion consists of three parts: the claim, the supporting metric, and the practical implication. For example: *"with the same budget of 16 measurements, the randomized search achieved an AP of 0.627 compared to 0.615 from the grid search, and 20 out of 20 random seeds beat the grid; therefore, I will default to randomized search whenever tuning more than two hyperparameters."*

### Exercise 12 — Three Conclusions

Write **three conclusions** following the format of the example above — claim, metric, and practical implication. Suggested topics:
- Why the computational cost of randomized search does not scale with the number of hyperparameters;
- What you began looking at in `cv_results_` beyond just the top-ranked row;
- Which metric you would publish in a report versus which one remains in development logs;
- What problem AutoML solved, and what critical tasks it left in your hands.

### **Answer - Exercise 12**

**1. On the immunity of randomized search to dimensionality:**
Because the probability of hitting a high-performance region depends exclusively on its relative volume fraction $p$ and the number of trials $n$, a randomized search requires exactly 15 trials to find a $15\%$ target volume region with over $90\%$ confidence, regardless of whether the hyperparameter space has 2 or 50 dimensions. This makes randomized search the default, scalable standard for complex, high-dimensional pipelines where grid search is computationally unfeasible.

**2. On reporting honest validation metrics:**
The optimistic selection bias in hyperparameter tuning leads to a naive `best_score_` ($0.6271$) that overestimates performance. To maintain academic and technical integrity, we must report the unbiased Nested CV score ($0.6354 \pm 0.1417$) as our expected generalization metric, keeping individual parameter sweeps in development logs.

**3. On the critical role of the scientist in automated workflows:**
While AutoML successfully scans multiple model families to find Logistic Regression as the top performer (AP of $0.7241$), it also fully exploits the leaked `popularidade` feature to yield an artificial $1.0000$ AP without any automated warning. This demonstrates that mathematical optimization is a powerful tool, but verifying data integrity and avoiding target leakage remains the irreplaceable responsibility of the data scientist.

---
**End.** You compared grid search and randomized search under the same budget, designed parameter distributions, evaluated a podium versus a plateau, responded to boundary-clinging optimal parameters, extracted the three honest evaluation scores, and set up a conditional multi-model AutoML search space — including a demonstration of how it optimistically exploits target leakage when data is not audited.

In **Challenge 3**, the goal is not just to produce the metric, but to justify it.